# Hybrid Transformer vs. Standard Transformer Comparison

This notebook compares the standard Probabilistic Transformer against the Hybrid Transformer which integrates an Ornstein-Uhlenbeck (OU) process to model residuals

## Methodology

1.  Standard transformer: predicting prices solely based on input features
2.  Hybrid Transformer:
    - Train Transformer for trend prediction
    - Model residuals (Actual - Predicted) using an OU process
    - Final forecast = transformer trend + OU process mean reversion

We run both models 10 times to reduce statistical variability and report average metrics (MAE, RMSE, MAPE, R2, Pinball Loss)

In [1]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

current_dir = Path.cwd()
project_root = None
if (current_dir / 'config.py').exists():
    project_root = str(current_dir)
elif (current_dir.parent / 'config.py').exists():
    project_root = str(current_dir.parent)

if project_root and project_root not in sys.path:
    sys.path.insert(0, project_root)

from models import ProbabilisticTransformer, HybridProbabilisticTransformer
from core.experiment_utils import (
    load_data, load_cache, save_cache, run_experiment, N_RUNS,
)

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

try:
    gpus = tf.config.experimental.list_physical_devices("GPU")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPUs detected: {len(gpus)}")
except Exception as e:
    print(f"GPU config failed: {e}")

2026-03-08 18:27:46.739393: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772990866.755470 1425959 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772990866.760443 1425959 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772990866.773104 1425959 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772990866.773119 1425959 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772990866.773121 1425959 computation_placer.cc:177] computation placer alr

GPUs detected: 1


In [2]:
RESULTS_DIR = Path(project_root) / "results" / "hybrid_comparison"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = RESULTS_DIR / "results.json"

In [3]:
# All metrics and caching handled by core.experiment_utils

In [4]:
data = load_data()


Data loaded  —  Train: 10366, Val: 1997, Test: 3722


In [5]:
cache = load_cache(CACHE_FILE)

run_experiment(
    ProbabilisticTransformer, "Standard Transformer", data,
    str(RESULTS_DIR), cache=cache, is_hybrid=False,
)

run_experiment(
    HybridProbabilisticTransformer, "Hybrid (Transformer+OU)", data,
    str(RESULTS_DIR), cache=cache, is_hybrid=True,
)

save_cache(CACHE_FILE, cache)

[Standard Transformer] found in cache — skipping.
[Hybrid (Transformer+OU)] found in cache — skipping.


In [6]:
results = [{"Model": k, **v} for k, v in cache.items()]
df_res = pd.DataFrame(results)
if not df_res.empty:
    display(df_res.sort_values("MAE"))
    df_res.to_csv(RESULTS_DIR / "comparison_summary.csv", index=False)

,Model,MAE,MSE,RMSE,MAPE,R2,PICP,MPIW,PINAW,IntervalScore,CRPS,NLL,Pinball_10,Pinball_50,Pinball_90,Avg_Pinball,training_time
1,Hybrid (Transformer+OU),20.195598,731.584318,27.034734,2008.837570,0.479984,0.936523,108.812365,0.302082,147.038630,14.538502,NaN,4.933939,9.982820,4.689742,6.535500,99.275769
0,Standard Transformer,20.305483,742.827745,27.218280,1977.640298,0.471992,0.898764,92.603129,0.257083,149.714597,14.607629,53.691174,4.976658,9.889398,4.739472,6.535176,101.593814
